# Data downloading file:

I keep only relevant columns (wind, radiaition, latitude, longitude, elevation). Only Swiss Cantons (don't keep measures in Liechtenstein). Dowload historical daily and monthly data. Merge all the montlhy and daily data using station key. Create a 'full_df'. 

In [6]:
import pandas as pd

debut = pd.Timestamp("1980-01-01", tz="UTC")
fin = pd.Timestamp("2026-01-01", tz="UTC")  # borne exclue

base = "https://data.geo.admin.ch/ch.meteoschweiz.ogd-smn"

# Métadonnées des stations
colonnes_stations = [
    "station_abbr",
    "station_name",
    "station_canton",
    "station_data_since",
    "station_coordinates_wgs84_lat",
    "station_coordinates_wgs84_lon",
    "station_height_masl",
]

stations = pd.read_csv(
    f"{base}/ogd-smn_meta_stations.csv",
    sep=";",
    encoding="cp1252",
    usecols=colonnes_stations,
)

stations = stations.rename(columns={
    "station_coordinates_wgs84_lat": "latitude",
    "station_coordinates_wgs84_lon": "longitude",
    "station_height_masl": "elevation",
})


cantons_suisses = {
    "AG", "AI", "AR", "BE", "BL", "BS", "FR", "GE", "GL", "GR",
    "JU", "LU", "NE", "NW", "OW", "SG", "SH", "SO", "SZ", "TG",
    "TI", "UR", "VD", "VS", "ZG", "ZH",
}
stations = stations[stations["station_canton"].isin(cantons_suisses)]

cles = ["station_abbr", "reference_timestamp"]
variables_jour = ["fkl010d0", "gre000d0"]
variables_mois = ["fkl010m0", "gre000m0"]
toutes_variables = variables_jour + variables_mois

donnees_stations = []

for abbr in stations["station_abbr"]:
    code = abbr.lower()
    dossier = f"{base}/{code}"

    # Fichiers historiques MétéoSuisse
    url_jour = f"{dossier}/ogd-smn_{code}_d_historical.csv"
    url_mois = f"{dossier}/ogd-smn_{code}_m.csv"

    try:
        jour = pd.read_csv(url_jour, sep=";", encoding="cp1252")
        mois = pd.read_csv(url_mois, sep=";", encoding="cp1252")
    except Exception as erreur:
        print(f"{abbr} ignorée : téléchargement impossible ({erreur})")
        continue

    # Harmoniser les noms de colonnes si nécessaire.
    for df in (jour, mois):
        df.columns = df.columns.str.replace("'", "", regex=False)
        df.rename(columns={"REFERENCE_TS": "reference_timestamp"}, inplace=True)

    # Ne garder que les colonnes utiles.
    jour = jour.reindex(columns=cles + variables_jour)
    mois = mois.reindex(columns=cles + variables_mois)

    # Réunir les données journalières et mensuelles.
    donnees = jour.merge(mois, on=cles, how="outer")

    donnees["reference_timestamp"] = pd.to_datetime(
        donnees["reference_timestamp"],
        format="%d.%m.%Y %H:%M",
        utc=True,
        errors="coerce",
    )

    donnees = donnees[
        donnees["reference_timestamp"].ge(debut)
        & donnees["reference_timestamp"].lt(fin)
    ]

    donnees_stations.append(donnees)

# Créer full_df et ajouter les métadonnées des stations.
full_df = pd.concat(donnees_stations, ignore_index=True)
full_df = full_df.merge(stations, on="station_abbr", how="left")

full_df = full_df[
    cles
    + toutes_variables
    + [
        "station_name",
        "station_canton",
        "station_data_since",
        "latitude",
        "longitude",
        "elevation",
    ]
].sort_values(cles, ignore_index=True)

full_df.head()

/var/folders/cy/bftv_ysd4yl_5fhdd6kdck500000gn/T/ipykernel_57329/3682303618.py:65: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={"REFERENCE_TS": "reference_timestamp"}, inplace=True)
/var/folders/cy/bftv_ysd4yl_5fhdd6kdck500000gn/T/ipykernel_57329/3682303618.py:65: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={"REFERENCE_TS": "reference_timestamp"}, inplace=True)
/var/folders/cy/bftv_ysd4yl_5fhdd6kdck500000gn/T/ipykernel_57329/3682303618.py:65: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsn

MLJ ignorée : téléchargement impossible (HTTP Error 403: Forbidden)


/var/folders/cy/bftv_ysd4yl_5fhdd6kdck500000gn/T/ipykernel_57329/3682303618.py:65: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={"REFERENCE_TS": "reference_timestamp"}, inplace=True)
/var/folders/cy/bftv_ysd4yl_5fhdd6kdck500000gn/T/ipykernel_57329/3682303618.py:65: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.rename(columns={"REFERENCE_TS": "reference_timestamp"}, inplace=True)
/var/folders/cy/bftv_ysd4yl_5fhdd6kdck500000gn/T/ipykernel_57329/3682303618.py:65: Pandas4Warning: The inplace keyword in DataFrame.rename is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsn

,station_abbr,reference_timestamp,fkl010d0,gre000d0,fkl010m0,gre000m0,station_name,station_canton,station_data_since,latitude,longitude,elevation
0,ABO,1980-01-01 00:00:00+00:00,0.7,NaN,0.6,NaN,Adelboden,BE,01.01.1901,46.491703,7.560703,1321.0
1,ABO,1980-01-02 00:00:00+00:00,0.2,NaN,NaN,NaN,Adelboden,BE,01.01.1901,46.491703,7.560703,1321.0
2,ABO,1980-01-03 00:00:00+00:00,0.3,NaN,NaN,NaN,Adelboden,BE,01.01.1901,46.491703,7.560703,1321.0
3,ABO,1980-01-04 00:00:00+00:00,0.7,NaN,NaN,NaN,Adelboden,BE,01.01.1901,46.491703,7.560703,1321.0
4,ABO,1980-01-05 00:00:00+00:00,0.0,NaN,NaN,NaN,Adelboden,BE,01.01.1901,46.491703,7.560703,1321.0


In [5]:
full_df.to_csv("/Users/lelub/Documents/4A_ENS_ARPE/WORKPATH/wind_solar/historic_desc_3/full_df.csv", index=False, encoding="utf-8")